# 04 - Model Training

Splits the data, trains two regression models and two classification models inside leakage-safe pipelines, runs cross-validation on the training split, and saves the best of each. This mirrors `src/train_model.py` exactly, cell by cell, for transparency.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))
os.chdir('..')  # so relative paths like 'data/raw/students.csv' resolve
import warnings
warnings.filterwarnings('ignore')


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score

from data_preprocessing import load_raw_data, clean_data, get_features_and_target
from feature_engineering import build_preprocessor, categorize_performance

RANDOM_STATE = 42
df = clean_data(load_raw_data())
X, y_reg = get_features_and_target(df)

clean_data: (505, 11) -> (500, 11) (removed 5 rows: duplicates / missing target)


## Train/test split (BEFORE any preprocessing is fit)

In [3]:
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=RANDOM_STATE)
y_clf_train = y_reg_train.apply(categorize_performance)
y_clf_test = y_reg_test.apply(categorize_performance)
print('Train:', X_train.shape, ' Test:', X_test.shape)

Train: (400, 6)  Test: (100, 6)


## Regression models: LinearRegression vs RandomForestRegressor

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reg_candidates = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
}
reg_results = {}
for name, model in reg_candidates.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', model)])
    cv = cross_val_score(pipe, X_train, y_reg_train, cv=kf, scoring='r2')
    pipe.fit(X_train, y_reg_train)
    preds = pipe.predict(X_test)
    reg_results[name] = {
        'pipeline': pipe,
        'cv_r2_mean': cv.mean(),
        'test_mae': mean_absolute_error(y_reg_test, preds),
        'test_rmse': np.sqrt(mean_squared_error(y_reg_test, preds)),
        'test_r2': r2_score(y_reg_test, preds),
    }
    print(name, '-> CV R2:', round(cv.mean(),4), '| Test R2:', round(reg_results[name]['test_r2'],4),
          '| Test RMSE:', round(reg_results[name]['test_rmse'],4))

LinearRegression -> CV R2: 0.662 | Test R2: 0.5483 | Test RMSE: 6.29


RandomForestRegressor -> CV R2: 0.5853 | Test R2: 0.4945 | Test RMSE: 6.6543


In [5]:
best_reg_name = max(reg_results, key=lambda n: reg_results[n]['test_r2'])
print('Selected regression model:', best_reg_name)

Selected regression model: LinearRegression


## Classification models: LogisticRegression vs RandomForestClassifier

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
clf_candidates = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}
clf_results = {}
for name, model in clf_candidates.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', model)])
    cv = cross_val_score(pipe, X_train, y_clf_train, cv=skf, scoring='accuracy')
    pipe.fit(X_train, y_clf_train)
    preds = pipe.predict(X_test)
    clf_results[name] = {
        'pipeline': pipe,
        'cv_acc_mean': cv.mean(),
        'test_accuracy': accuracy_score(y_clf_test, preds),
        'test_f1': f1_score(y_clf_test, preds, average='weighted', zero_division=0),
    }
    print(name, '-> CV Acc:', round(cv.mean(),4), '| Test Acc:', round(clf_results[name]['test_accuracy'],4),
          '| Test F1:', round(clf_results[name]['test_f1'],4))

LogisticRegression -> CV Acc: 0.715 | Test Acc: 0.76 | Test F1: 0.7594


RandomForestClassifier -> CV Acc: 0.7075 | Test Acc: 0.74 | Test F1: 0.7242


In [7]:
best_clf_name = max(clf_results, key=lambda n: clf_results[n]['test_f1'])
print('Selected classification model:', best_clf_name)

Selected classification model: LogisticRegression


## Save the winning pipelines

Both winning pipelines are saved together in a single joblib artifact by `src/train_model.py` (`models/student_performance_model.joblib`), so training and inference always use identical preprocessing.

In [8]:
import joblib, os
os.makedirs('models', exist_ok=True)
artifact = {
    'regressor': reg_results[best_reg_name]['pipeline'],
    'regressor_name': best_reg_name,
    'classifier': clf_results[best_clf_name]['pipeline'],
    'classifier_name': best_clf_name,
    'performance_categories': ['Needs Improvement', 'Average', 'Good', 'Excellent'],
    'feature_columns': list(X.columns),
    'random_state': RANDOM_STATE,
}
joblib.dump(artifact, 'models/student_performance_model.joblib')
print('Saved to models/student_performance_model.joblib')

Saved to models/student_performance_model.joblib
